In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
df =pd.read_excel("post_shock_refill_buy_trades.xlsx")
df.head()

,symbol,minute,twap_time,strategy_time,twap_buy_price,strategy_buy_price,buy_improvement,pct,bp
0,AMZN,1900-01-01 09:30:00,09:30:59.726000,09:30:01.290000,224.35,223.86,0.49,0.002184,0.218409
1,AMZN,1900-01-01 09:31:00,09:31:59.124000,09:31:50.319000,223.70,223.60,0.10,0.000447,0.044703
2,AMZN,1900-01-01 09:32:00,09:32:58.437000,09:32:57.568000,223.82,223.76,0.06,0.000268,0.026807
3,AMZN,1900-01-01 09:33:00,09:33:59.626000,09:33:40.718000,224.20,223.86,0.34,0.001517,0.151650
4,AMZN,1900-01-01 09:34:00,09:34:59.913000,09:34:58.770000,224.32,224.32,0.00,0.000000,0.000000


In [3]:
twap_df = pd.read_csv("../backtest/twap_buy_execution.csv")
twap_df.head()

,stock_name,minute,buy_time,buy_price,strategy
0,AMZN,1900-01-01 12:39:00,1900-01-01 12:39:05.651,222.45,TWAP_Buy
1,AMZN,1900-01-01 12:40:00,1900-01-01 12:40:01.032,222.54,TWAP_Buy
2,AMZN,1900-01-01 12:41:00,1900-01-01 12:41:00.098,222.54,TWAP_Buy
3,AMZN,1900-01-01 12:42:00,1900-01-01 12:42:00.510,222.43,TWAP_Buy
4,AMZN,1900-01-01 12:43:00,1900-01-01 12:43:00.045,222.62,TWAP_Buy


In [5]:
# rename columns
df = df.rename(columns={"symbol": "stock"})
twap_df = twap_df.rename(columns={"stock_name": "stock"})

# make minute columns the same dtype
df["minute"] = pd.to_datetime(df["minute"])
twap_df["minute"] = pd.to_datetime(twap_df["minute"])

# merge
merged = twap_df.merge(
    df[["stock", "minute", "strategy_buy_price"]],
    on=["stock", "minute"],
    how="inner"
)

# compute row-level improvement
merged["average_improvement"] = merged["buy_price"] - merged["strategy_buy_price"]

# aggregate to the format you want
result = (
    merged.groupby("stock", as_index=False)["average_improvement"]
    .mean()
)

print(result)

  stock  average_improvement
0  AMZN             0.084815
1  GOOG             0.127531
2  INTC             0.004444
3  MSFT             0.006667


In [6]:
result.to_csv("charlie.csv", index=False)

In [7]:
oracle_df = pd.read_csv("oracle_buy_execution.csv")
oracle_df.head()

,stock,average_improvement
0,AMZN,0.096173
1,GOOG,0.169753
2,INTC,0.006914
3,MSFT,0.008519


In [8]:
oracle_df['average_improvement'] - result['average_improvement']

0    0.011358
1    0.042222
2    0.002469
3    0.001852
Name: average_improvement, dtype: float64